# Overview

This example uploads one scanning probe microscopy run to the platform: the run folder becomes a Sample Set with one Sample per measured position, a Measurement Set with one Measurement per Sample and its Setup, the run's records as files, and one hysteresis-loop Property per Sample.
A run folder is what the instrument exports — `summary.json` with the recipe, the session and one record per measured point, and `loops/` with the raw curves — and re-running the notebook adds only what is missing.

## Install the API client

The samples, measurements and files endpoints are not released yet, so the client is installed from its branch until it merges. Restart the kernel after this cell.

In [1]:
%pip install -r requirements.txt

  error: subprocess-exited-with-error
  
  × git version did not run successfully.
  │ exit code: 1
  ╰─> [2 lines of output]
      xcrun: error: invalid active developer path (/Library/Developer/CommandLineTools), missing xcrun at: /Library/Developer/CommandLineTools/usr/bin/xcrun
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
ERROR: Failed to build 'mat3ra-api-client' when git version
Note: you may need to restart the kernel to use updated packages.


## Set Parameters

- **HOST**: platform the run is uploaded to
- **RUN_DIR**: the run folder beside this notebook — what the instrument exports, with `summary.json` and `loops/` inside it
- **PHYSICAL_ID**: the identifier written on the physical piece the measured positions are part of — every Sample carries it
- **ACCOUNT_SLUG**: account the data belongs to, empty for the default account
- **FILES**: which files to upload per measurement

In [2]:
import urllib.parse

HOST = "https://alphafilm.mat3ra.com"

# NOTE: generate at https://alphafilm.mat3ra.com/demo/preferences API Tokens
ACCOUNT_ID = "pR5gsAQJrJYJuapM3"  # from Preferences
AUTH_TOKEN = "gFEku1YpH4gTF57g0nYUj2C_UYM5brGSZXjSoCywI-H"  # the API token

RUN_DIR = "/Users/mat3ra/code/work/SOF-8050/data/From_UTK"  # folder with summary.json, records etc
PHYSICAL_ID = "test-2"  # the physical wafer's identifier (e.g. PDAC_COM5_01448)
FILES = ["records", "loops"]  # Which folders to upload as files per measurement.

url = urllib.parse.urlsplit(HOST)
address = {
    "host": url.hostname,
    "port": url.port or (443 if url.scheme == "https" else 80),
    "secure": url.scheme == "https",
}

## Authenticate and initialize API client

### Authenticate
Authenticate in the browser (OIDC device flow) or via JupyterLite host injection. Credentials are stored in environment variables.

### Initialize API client
Create an authenticated API client and resolve the owner account ID.

In [3]:
from mat3ra.notebooks_utils.auth import authenticate
import os

os.environ["ACCOUNT_ID"] = ACCOUNT_ID
os.environ["AUTH_TOKEN"] = AUTH_TOKEN


# NOTE: uncomment to login with OIDC interactively
# os.environ["API_HOST"] = address["host"]
# os.environ["API_PORT"] = str(address["port"])
# os.environ["API_SECURE"] = str(address["secure"])
# await authenticate()

In [4]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate(**address)

# Imports

In [5]:
from pathlib import Path

from parse_utk import parse
from run_document import load, serialize
from upload_run import account_id, upload

ModuleNotFoundError: No module named 'mat3ra.esse.models.sample'

## Parse the run folder

Read the run folder into the documents the platform stores. Nothing is uploaded yet.

In [ ]:
# reading the run folder and writing the run document: nothing here talks to the platform
parsed = parse(Path(RUN_DIR), PHYSICAL_ID)
document_path = serialize(parsed, "parsed")
run = load(document_path)
file_count = sum(len(files) for files in run["files"].values())
print(
    f"{run['physicalId']}: {len(run['samples'])} samples (ordered set) · run {run['run']}: "
    f"{len(run['measurements'])} measurements (ordered set, one per sample) · {len(run['set_files'])} set files "
    f"-> {file_count} measurement files · {len(run['properties'])} samples with a combined loop "
    f"· no curves: {len(run['skipped'])} samples"
)
print("run document:", document_path)

## Upload the run

Create the Sample Set and its Samples, the Measurement Set and one Measurement per Sample, the files and the loop Properties.

In [ ]:
upload(client, run, files=FILES)

## Find the run in the web app

The run is a folder in the account's Measurements tab, named after the run.

In [ ]:
print(f"Open {HOST}, your account's Measurements tab: {run['run']}")